In [0]:
SILVER_PATH = "/Volumes/workspace/legal_data/silver/legal_sections/"

In [0]:
import re
import uuid
from pyspark.sql.functions import col
from pyspark.sql import Row

In [0]:
BRONZE_PATH = "/Volumes/workspace/legal_data/bronze/legal_documents/"

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

bronze_df = bronze_df.filter(col("status") == "success")
bronze_df.count()

In [0]:
def clean_legal_text(text):
    if not text:
        return ""

    # remove page numbers
    text = re.sub(r'Page\s*\d+', ' ', text, flags=re.IGNORECASE)

    # remove headers/footers repeated in uppercase
    text = re.sub(r'\b[A-Z ]{6,}\b', ' ', text)

    # remove bullet symbols
    text = re.sub(r'[•●■▪]', ' ', text)

    # normalize whitespace
    text = re.sub(r'\n\s*\n', '\n', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [0]:
def extract_act_name(file_name, category):
    name = file_name.replace(".pdf", "")
    name = name.replace("_", " ").replace("-", " ")

    if "constitution" in name.lower():
        return "Constitution of India"

    return name.title()

In [0]:
LEGAL_PATTERNS = [
    r'(Article\s+\d+[A-Za-z\-]*)',
    r'(Section\s+\d+[A-Za-z\-]*)',
    r'(Sec\.?\s*\d+[A-Za-z\-]*)',
    r'(Rule\s+\d+[A-Za-z\-]*)',
    r'(Chapter\s+[IVXLC]+)',
]

In [0]:
def split_legal_sections(text):
    if not text:
        return []

    pattern = "|".join(LEGAL_PATTERNS)
    parts = re.split(pattern, text)

    sections = []

    for i in range(1, len(parts), 2):
        header = parts[i]
        content = parts[i+1] if i+1 < len(parts) else ""

        # ✅ SAFETY CHECKS
        if header is None:
            continue

        header = str(header).strip()
        content = str(content).strip() if content else ""

        if header:  # ensure header not empty
            sections.append((header, content))

    return sections

In [0]:
def split_paragraphs(text, max_chars=1200):
    paragraphs = re.split(r'\n+', text)

    chunks = []
    current = ""

    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += " " + para
        else:
            chunks.append(current.strip())
            current = para

    if current:
        chunks.append(current.strip())

    return chunks

In [0]:
records = []

for row in bronze_df.collect():

    cleaned_text = clean_legal_text(row.raw_text)
    act_name = extract_act_name(row.file_name, row.category)

    sections = split_legal_sections(cleaned_text)

    # CASE 1: Sections found
    if sections:
        for header, body in sections:
            records.append(Row(
                section_id=str(uuid.uuid4()),
                doc_id=row.doc_id,
                file_name=row.file_name,
                category=row.category,
                act_name=act_name,
                section_number=header,
                section_title=header,
                section_text=body,
                source_path=row.file_path,
                created_at=row.ingestion_time
            ))

    # CASE 2: No sections → fallback paragraph chunks
    else:
        paragraphs = split_paragraphs(cleaned_text)

        for para in paragraphs:
            records.append(Row(
                section_id=str(uuid.uuid4()),
                doc_id=row.doc_id,
                file_name=row.file_name,
                category=row.category,
                act_name=act_name,
                section_number=None,
                section_title=None,
                section_text=para,
                source_path=row.file_path,
                created_at=row.ingestion_time
            ))

In [0]:
silver_df = spark.createDataFrame(records)

silver_df.display()

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .save(SILVER_PATH)

In [0]:
%sql
CREATE TABLE workspace.default.silver_legal_sections (
  section_id STRING,
  doc_id STRING,
  file_name STRING,
  category STRING,
  act_name STRING,
  section_number STRING,
  section_title STRING,
  section_text STRING,
  source_path STRING,
  created_at TIMESTAMP
)
USING DELTA;

In [0]:
silver_df.write \
    .mode("append") \
    .saveAsTable("workspace.default.silver_legal_sections")

In [0]:
silver_df.count()

In [0]:
silver_df.select('file_name').distinct().show()


In [0]:
silver_df.groupBy("file_name").count().display()

In [0]:
silver_df.select("section_number","section_title").limit(10).display()

In [0]:
silver_df.select("section_text").limit(5).display()

In [0]:
%sql
select count(*) from workspace.default.silver_legal_sections

In [0]:
%sql
select file_name, category, act_name from workspace.default.silver_legal_sections limit 25